In [4]:
from MotorInversion import MotorInversion
import pandas as pd
import importlib
import itertools
from datetime import datetime, date
from io import StringIO
import matplotlib.pyplot as plt

from UniversoActivos import UniversoActivosEstatico, UniversoActivosDinamico
from ProveedorDatos import YFinanceProvider
from VariablesTransformation import FeatureEngineer
import Modelos
Modelos = importlib.reload(Modelos)
import Estrategia
importlib.reload(Estrategia)
from Backtest import BacktestEngine
import requests

RandomForestModel = Modelos.RandomForestModel
XGBoostModel = Modelos.XGBoostModel

EstrategiaMLEquiponderada = Estrategia.EstrategiaMLEquiponderada
EstrategiaMLMinVarAlphaTilt = Estrategia.EstrategiaMLMinVarAlphaTilt
EstrategiaMLMonteCarlo = Estrategia.EstrategiaMLMonteCarlo
EstrategiaMLEquiponderadaMacro = Estrategia.EstrategiaMLEquiponderadaMacro

In [6]:
def get_eurostoxx50_tickers():
    url = 'https://en.wikipedia.org/wiki/EURO_STOXX_50'
    headers = {"User-Agent": "Mozilla/5.0 ..."}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text), flavor='bs4')
    df = next(t for t in tables if 'Ticker' in t.columns)
    return df['Ticker'].tolist()

tickers = get_eurostoxx50_tickers()
print(len(tickers))
print(tickers)

50
['ADS.DE', 'ADYEN.AS', 'AD.AS', 'AI.PA', 'AIR.PA', 'ALV.DE', 'ABI.BR', 'ARGX.BR', 'ASML.AS', 'CS.PA', 'BAS.DE', 'BAYN.DE', 'BBVA.MC', 'SAN.MC', 'BMW.DE', 'BNP.PA', 'BN.PA', 'DBK.DE', 'DB1.DE', 'DHL.DE', 'DTE.DE', 'ENEL.MI', 'ENI.MI', 'EL.PA', 'RACE.MI', 'RMS.PA', 'IBE.MC', 'ITX.MC', 'IFX.DE', 'INGA.AS', 'ISP.MI', 'OR.PA', 'MC.PA', 'MBG.DE', 'MUV2.DE', 'NDA-FI.HE', 'PRX.AS', 'RHM.DE', 'SAF.PA', 'SGO.PA', 'SAN.PA', 'SAP.DE', 'SU.PA', 'SIE.DE', 'ENR.DE', 'TTE.PA', 'DG.PA', 'UCG.MI', 'VOW.DE', 'WKL.AS']


In [7]:
universo   = UniversoActivosEstatico(tickers)
fe = FeatureEngineer(criterio=10, ticker_indice="^STOXX50E")
modelo = XGBoostModel(random_state=42, n_splits=3)
estrategia = EstrategiaMLMonteCarlo(modelo=modelo, n_activos_obj=15, umbral_salida=22,
                                    n_simulaciones=5000, peso_min=0.02, peso_max=0.15)

motor = MotorInversion(
    universo       = universo,
    feature_engineer = fe,
    estrategia     = estrategia,
    proveedor_cls  = YFinanceProvider,
    capital_total  = 10000000,
    estado_path    = "./mi_cartera",   # carpeta donde guarda los CSV
    len_ventana = 4
)

In [4]:
# Código para reentrenar el modelo por si cambiamos cosas en el codigo
motor._ultimo_train = None
motor._reentrenar_si_toca(date(2026, 3, 4))
motor._guardar_modelo()

[Train] 2021-03-05 → 2026-02-20 | AUC=0.5484 | {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 150, 'scale_pos_weight': 3, 'subsample': 0.7}


In [ ]:
# Cada viernes ejecutas esto:
señales = motor.ejecutar()

[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  51 of 51 completed


Compras hechas a fecha 2026-03-13 00:00:00
  Ticker Accion  Cantidad      Precio     CT  Precio_Ejecutado
 ASML.AS COMPRA       564 1180.400024 1.1804         1181.5804
  ENR.DE COMPRA      4362  152.649994 0.1526          152.8026
 BBVA.MC COMPRA     36623   18.184999 0.0182           18.2032
  ADS.DE COMPRA      4716  141.199997 0.1412          141.3412
  PRX.AS COMPRA     14659   45.430000 0.0454           45.4754
  SAP.DE COMPRA      3987  167.020004 0.1670          167.1870
   EL.PA COMPRA      3159  210.800003 0.2108          211.0108
 RACE.MI COMPRA      2276  292.600006 0.2926          292.8926
  WKL.AS COMPRA      9919   67.139999 0.0671           67.2071
 BAYN.DE COMPRA     17020   39.130001 0.0391           39.1691
ADYEN.AS COMPRA       726  917.299988 0.9173          918.2173
  DBK.DE COMPRA     25909   25.705000 0.0257           25.7307
  RHM.DE COMPRA       429 1550.500000 1.5505         1552.0505
 ARGX.BR COMPRA      1081  616.000000 0.6160          616.6160
  AIR.PA COM

Viernes 20 de marzo de 2026

In [5]:
fecha = date(2026, 3, 20)
señales = motor.ejecutar(fecha)

c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
   EL.PA COMPRA        74  194.750000 0.097375        194.847375
ADYEN.AS COMPRA         3  863.500000 0.431750        863.931750
  PRX.AS COMPRA      1077   40.020000 0.020010         40.040010
  ENR.DE COMPRA       112  140.750000 0.070375        140.820375
  ADS.DE COMPRA         1  133.500000 0.066750        133.566750
 RACE.MI COMPRA        24  273.799988 0.136900        273.936888
  SAP.DE COMPRA       107  153.820007 0.076910        153.896917
  RMS.PA COMPRA       380 1656.000000 0.828000       1656.828000
  ENI.MI COMPRA     26650   23.620001 0.011810         23.631811
  SGO.PA COMPRA      9246   68.080002 0.034040         68.114042
  WKL.AS  VENTA       294   65.440002 0.032720         65.407282
 ARGX.BR  VENTA         4  584.799988 0.292400        584.507588
 BAYN.DE  VENTA       612   38.384998 0.019192         38.365806
  DBK.DE  VENTA       472   24.760000 0.012380         24.747620
 BBVA.MC  VENTA     36623

Viernes 27 de marzo de 2026

In [ ]:
fecha = date(2026, 3, 27)
señales = motor.ejecutar(fecha)

  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  IFX.DE COMPRA      5846   37.430000 0.018715         37.448715
  ENI.MI COMPRA     34714   23.924999 0.011962         23.936962
 ASML.AS COMPRA       279 1146.400024 0.573200       1146.973224
  RHM.DE COMPRA       221 1379.500000 0.689750       1380.189750
  TTE.PA COMPRA     18700   78.489998 0.039245         78.529243
  UCG.MI COMPRA     12173   60.220001 0.030110         60.250111
  SAF.PA COMPRA      1724  278.399994 0.139200        278.539194
 INGA.AS COMPRA     34709   21.719999 0.010860         21.730859
  SAN.MC COMPRA     56041    9.402000 0.004701          9.406701
 BAYN.DE  VENTA      1609   38.259998 0.019130         38.240868
  SGO.PA  VENTA      1553   69.120003 0.034560         69.085443
  DBK.DE  VENTA     13938   24.915001 0.012458         24.902543
  ADS.DE  VENTA      2845  132.050003 0.066025        131.983978
  PRX.AS  VENTA     15736   38.720001 0.019360         38.700641
   EL.PA  VENTA      3233

Viernes 3 de abril de 2026

In [8]:
fecha = date(2026, 4, 3)
señales = motor.ejecutar(fecha)


1 Failed download:
['ABI.BR']: OperationalError('database is locked')


[Señales] 2026-04-03 | Pesos objetivo: {'ADS.DE': 0.039212258605359214, 'ADYEN.AS': 0.025893622067794997, 'ASML.AS': 0.13792986724016687, 'BAYN.DE': 0.09055288399939496, 'BBVA.MC': 0.07347758338918811, 'DBK.DE': 0.03365522007560314, 'ENI.MI': 0.12345330697614293, 'ENR.DE': 0.02612472421507548, 'IFX.DE': 0.04816259175438095, 'INGA.AS': 0.06813066522371464, 'RACE.MI': 0.02383311062131374, 'SAF.PA': 0.05836475231588556, 'SAN.MC': 0.05241753137401491, 'SGO.PA': 0.02786433914828268, 'TTE.PA': 0.17092754299368176}
Valor de la cartera antes de ejecutar señales: 9701263.39€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
 BAYN.DE COMPRA      7328   39.695000 0.019847         39.714847
 BBVA.MC COMPRA     37957   18.770000 0.009385         18.779385
 RACE.MI COMPRA       782  295.500000 0.147750        295.647750
  SAF.PA COMPRA       246  287.299988 0.143650        287.443638
 ASML.AS COMPRA       314 1161.000000 0.580500       1161.580500
  ADS.DE COMPRA       947  134.899994

In [9]:
import yfinance as yf

# Definir el ticker (ejemplo: Apple)
ticker = "WKL.AS"

# Descargar datos con periodo de 1 día e intervalo de 1 minuto
data = yf.download(ticker, period="5d", interval="1d")

# Mostrar las últimas filas
print(data.tail())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open   Volume
Ticker         WKL.AS     WKL.AS     WKL.AS     WKL.AS   WKL.AS
Date                                                           
2026-03-23  63.820000  66.980003  63.200001  64.459999  1139675
2026-03-24  62.560001  64.879997  62.240002  64.220001   933663
2026-03-25  62.740002  63.459999  61.840000  63.279999   917136
2026-03-26  63.959999  64.300003  61.919998  62.299999  1110580
2026-03-27  62.320000  64.699997  62.320000  64.419998   947115
